In [3]:
from pathlib import Path
from collections import defaultdict, Counter

import polars as pl

In [5]:
#1.path
CURRENT_DIR = Path.cwd()

PROJECT_ROOT = CURRENT_DIR.parent

DATA_ROOT = PROJECT_ROOT / "tennis_data"

EXTRACT_ROOT = DATA_ROOT / "extracted"



print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA ROOT:", DATA_ROOT)
print("EXTRACT ROOT:", EXTRACT_ROOT)

PROJECT_ROOT: /Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis
DATA ROOT: /Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data
EXTRACT ROOT: /Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted


In [6]:
# 2. Find all Season Parquet files
season_files = sorted(
    EXTRACT_ROOT.glob(
        "*/season_*.parquet"
    )
)

print("Number of Season files:",len(season_files))

Number of Season files: 35671


In [7]:
# 2. Display some Season files

# Display first 10 season files
# to verify that the correct files were found

for file in season_files[:10]:
    print(file)

/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/season_11974053.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/season_11974066.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/season_11998445.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/season_11998446.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/season_11998447.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/season_11998448.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/extracted/20240201/season_11998449.parquet
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/ext

In [8]:
# 4. Check schema of one Season file

# Read the first Season parquet file

test_season_df = pl.read_parquet(
    season_files[0]
)


# Display data

print(test_season_df)


# Display schema

print("\nSchema:")
print(test_season_df.schema)

shape: (1, 4)
┌──────────┬───────────┬────────────────┬──────┐
│ match_id ┆ season_id ┆ name           ┆ year │
│ ---      ┆ ---       ┆ ---            ┆ ---  │
│ i64      ┆ i64       ┆ str            ┆ i64  │
╞══════════╪═══════════╪════════════════╪══════╡
│ 11974053 ┆ 55584     ┆ Davis Cup 2024 ┆ 2024 │
└──────────┴───────────┴────────────────┴──────┘

Schema:
Schema({'match_id': Int64, 'season_id': Int64, 'name': String, 'year': Int64})


In [9]:
# 5. Check schemas of all Season files

schemas = Counter()


# Read every Season parquet file and store its schema

for file in season_files:

    df = pl.read_parquet(file)

    schema_tuple = tuple(
        df.schema.items()
    )

    schemas[schema_tuple] += 1


# Display number of different schemas

print("Number of different schemas:",len(schemas))

Number of different schemas: 1


In [10]:
# 6. Read and process all Season files


# Store processed Season DataFrames
season_frames = []


# Process every Season parquet file

for file in season_files:

    df = pl.read_parquet(file)


    # Extract snapshot date from folder name
    snapshot_date = file.parent.name


    # Add snapshot_date column

    df = df.with_columns(
        pl.lit(snapshot_date)
        .str.strptime(
            pl.Date,
            "%Y%m%d"
        )
        .alias("snapshot_date")
    )


    season_frames.append(df)


print("Number of processed Season files:",len(season_frames))

Number of processed Season files: 35671


In [11]:
# 7. Concatenate all Season DataFrames

# Combine all processed Season DataFrames vertically.
# Each row represents a Season record from a snapshot.

season_snapshot = pl.concat(
    season_frames,
    how="vertical"
)


print("Final Season dataset shape:",season_snapshot.shape)

Final Season dataset shape: (35671, 5)


DATA CLEANING

In [12]:
# 8. Check missing values in Season dataset
# Count null values in each column

print(season_snapshot.null_count())

shape: (1, 5)
┌──────────┬───────────┬──────┬──────┬───────────────┐
│ match_id ┆ season_id ┆ name ┆ year ┆ snapshot_date │
│ ---      ┆ ---       ┆ ---  ┆ ---  ┆ ---           │
│ u32      ┆ u32       ┆ u32  ┆ u32  ┆ u32           │
╞══════════╪═══════════╪══════╪══════╪═══════════════╡
│ 0        ┆ 0         ┆ 0    ┆ 0    ┆ 0             │
└──────────┴───────────┴──────┴──────┴───────────────┘


In [13]:
# 9. Check duplicated rows
# Check if there are completely duplicated rows

duplicate_count = (
    season_snapshot
    .is_duplicated()
    .sum()
)


print("Number of duplicated rows:",duplicate_count)

Number of duplicated rows: 0


In [14]:
# 10. Check match IDs with multiple snapshots


season_snapshot_counts = (
    season_snapshot
    .group_by("match_id")
    .agg(
        pl.col("snapshot_date")
        .n_unique()
        .alias("number_of_snapshots")
    )
)


# Find matches appearing in more than one snapshot

multiple_season_snapshots = (
    season_snapshot_counts
    .filter(
        pl.col("number_of_snapshots") > 1
    )
)


print("Match IDs with multiple snapshots:",multiple_season_snapshots.shape[0])


multiple_season_snapshots.head(10)

Match IDs with multiple snapshots: 16343


match_id,number_of_snapshots
i64,u32
12039188,3
12046247,2
12087938,2
12052321,2
12063325,2
12077342,2
12024769,2
12063554,3
12052684,2


In [15]:
# 11. Check if Season information changes between snapshots

season_changes = (
    season_snapshot
    .group_by("match_id")
    .agg(
        pl.col("season_id")
        .n_unique()
        .alias("different_season_ids"),

        pl.col("name")
        .n_unique()
        .alias("different_names"),

        pl.col("year")
        .n_unique()
        .alias("different_years")
    )
)


# Find matches where season information changed

changed_seasons = (
    season_changes
    .filter(
        (pl.col("different_season_ids") > 1)
        |
        (pl.col("different_names") > 1)
        |
        (pl.col("different_years") > 1)
    )
)


print("Number of matches with Season changes:",changed_seasons.shape[0])


changed_seasons.head(10)

Number of matches with Season changes: 0


match_id,different_season_ids,different_names,different_years
i64,u32,u32,u32


In [16]:
# 12. Check season_id consistency

season_id_check = (
    season_snapshot
    .group_by("season_id")
    .agg(
        pl.col("name")
        .n_unique()
        .alias("unique_names"),

        pl.col("year")
        .n_unique()
        .alias("unique_years")
    )
)


print("Season IDs with inconsistent information:")


print(
    season_id_check
    .filter(
        (pl.col("unique_names") > 1)
        |
        (pl.col("unique_years") > 1)
    )
)

Season IDs with inconsistent information:
shape: (0, 3)
┌───────────┬──────────────┬──────────────┐
│ season_id ┆ unique_names ┆ unique_years │
│ ---       ┆ ---          ┆ ---          │
│ i64       ┆ u32          ┆ u32          │
╞═══════════╪══════════════╪══════════════╡
└───────────┴──────────────┴──────────────┘


In [17]:
#13. Save cleaned Season dataset

# Create Data folder if it does not exist

clean_path = DATA_ROOT / "Data"

clean_path.mkdir(
    parents=True,
    exist_ok=True
)


# Save cleaned Season dataset

season_snapshot.write_parquet(
    clean_path / "season_clean.parquet"
)


print("Saved successfully:")

print(clean_path / "season_clean.parquet")

Saved successfully:
/Users/macbook/Desktop/ data_analysis_tannis projects/tennis_data_analysis/tennis_data/Data/season_clean.parquet


In [22]:
#14. Verify saved Season file

season_clean = pl.read_parquet(
    clean_path / "season_clean.parquet"
)


print(season_clean.shape)
print("*"*90)
print(season_clean.schema)
print("*"*90)

season_clean.head(20)

(35671, 5)
******************************************************************************************
Schema({'match_id': Int64, 'season_id': Int64, 'name': String, 'year': Int64, 'snapshot_date': Date})
******************************************************************************************


match_id,season_id,name,year,snapshot_date
i64,i64,str,i64,date
11974053,55584,"""Davis Cup 2024""",2024,2024-02-01
11974066,55584,"""Davis Cup 2024""",2024,2024-02-01
11998445,55448,"""ATP Montpellier, France Men Si…",2024,2024-02-01
11998446,55448,"""ATP Montpellier, France Men Si…",2024,2024-02-01
11998447,55448,"""ATP Montpellier, France Men Si…",2024,2024-02-01
…,…,…,…,…
11998672,56670,"""ATP Challenger Burnie, Austral…",2024,2024-02-01
11998674,56670,"""ATP Challenger Burnie, Austral…",2024,2024-02-01
11998675,56670,"""ATP Challenger Burnie, Austral…",2024,2024-02-01
